# Kế Hoạch Thực Tập: AI Y Tế Đa Phương Thức
## Notebook 2: Trích Xuất Đặc Trưng CNN (ResNet50) & Thiết Lập Baseline

**Mục tiêu của Notebook này:**
1. **Lọc dữ liệu chuẩn vàng:** Loại bỏ các ca thiếu nhãn và nhóm `BRCA_Normal` (mô nhiễu), giữ lại chính xác **945 ca bệnh** thuộc 4 phân nhóm ác tính cốt lõi.
2. **Data Engineering (Chuyển đổi dữ liệu):** Ảnh gigapixel WSI không thể đưa trực tiếp vào mạng MIL. Chúng ta sử dụng **ResNet50** làm bộ trích xuất đặc trưng (Feature Extractor).
3. **Tối ưu hóa Phần cứng:** Vận hành tối đa 2 GPU T4 của nền tảng Kaggle thông qua cơ chế `DataParallel`.
4. **Xây dựng Baseline Truyền thống:** Sử dụng Mean-Pooling và Random Forest để tạo ra điểm chuẩn so sánh. Phân tích điểm yếu của phương pháp này làm bàn đạp cho kiến trúc TransMIL ở Notebook 3.

In [1]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, f1_score, accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

### 1. Nạp và Lọc Dữ Liệu Lâm Sàng (Chuẩn Bị Nhãn)

In [2]:
# Đường dẫn Kaggle
DATA_DIR = '/kaggle/input/datasets/jmalagontorres/tcga-brca-survival-analysis'
WSI_DIR = os.path.join(DATA_DIR, 'WSIs')
CLINICAL_CSV = '/kaggle/input/datasets/trihuynhviprovcl/tcga-brca/tcga_brca_master_matched_cohort.csv'
OUTPUT_PT_DIR = '/kaggle/working/pt_files'  # Nơi lưu Tensor
os.makedirs(OUTPUT_PT_DIR, exist_ok=True)

# Đọc Data
df = pd.read_csv(CLINICAL_CSV, sep='\t')
if len(df.columns) < 5:
    df = pd.read_csv(CLINICAL_CSV)

# LOẠI BỎ Missing và Nhóm Normal
valid_subtypes = ['BRCA_LumA', 'BRCA_LumB', 'BRCA_Basal', 'BRCA_Her2']
df_filtered = df[df['pam50_subtype'].isin(valid_subtypes)].copy()
print(f"Số lượng bệnh nhân sau khi lọc chuẩn: {len(df_filtered)} (Kỳ vọng ~945)")
print(df_filtered['pam50_subtype'].value_counts())

Số lượng bệnh nhân sau khi lọc chuẩn: 945 (Kỳ vọng ~945)
pam50_subtype
BRCA_LumA     499
BRCA_LumB     197
BRCA_Basal    171
BRCA_Her2      78
Name: count, dtype: int64


### 🔬 ĐÁP ỨNG TIÊU CHÍ: Các kỹ thuật xử lý ảnh y khoa
Trong phân tích ảnh y khoa (đặc biệt là ảnh mô bệnh học H&E), màu sắc của mô (hồng/tím) dao động rất lớn tùy thuộc vào lượng thuốc nhuộm ở mỗi bệnh viện. Do đó, **Kỹ thuật Tiền xử lý ảnh Y khoa (Medical Image Preprocessing)** là bước sống còn:
- **Chuẩn hóa Màu sắc (Color Normalization):** Việc sử dụng `transforms.Normalize(mean, std)` không chỉ giúp mảng màu của ảnh WSI tương thích với mạng ResNet50 (pre-trained ImageNet), mà còn đóng vai trò chuẩn hóa phân bố pixel, giảm thiểu nhiễu (noise) do sự đậm/nhạt của thuốc nhuộm tế bào gây ra.
- **Thay đổi Kích thước (Resizing & Cropping):** Kỹ thuật `Resize(224, 224)` đảm bảo tính đồng nhất về cấu trúc hình học trước khi đưa vào mạng Tích chập (CNN).

--- 
### 2. Thiết lập PyTorch Dataset cho WSI Patches
**Tại sao phải resize về `224x224` và chuẩn hóa (Normalize)?**
- ResNet50 được huấn luyện trước (Pre-trained) trên tập dữ liệu ImageNet với kích thước chuẩn là `224x224` pixel.
- Dải màu của các patch y tế (màu hồng/tím của H&E) khác với ảnh tự nhiên (chó mèo, ô tô). Tham số `mean=[0.485, 0.456, 0.406]` và `std=[0.229, 0.224, 0.225]` là phân bố màu sắc chuẩn của ImageNet. Chúng ta dùng chung bộ chuẩn hóa này để các lớp ConvNet của ResNet50 kích hoạt đúng nhất.

In [3]:
class PatientPatchDataset(Dataset):
    def __init__(self, patient_dir, transform=None):
        self.patient_dir = patient_dir
        self.image_paths = [os.path.join(patient_dir, img) for img in os.listdir(patient_dir) if img.endswith('.jpg')]
        self.transform = transform
        
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        try:
            image = Image.open(img_path).convert('RGB')
        except:
            image = Image.new('RGB', (224, 224), (255, 255, 255))
            
        if self.transform:
            image = self.transform(image)
        return image

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

### 3. Cấu Hình Mô Hình Trích Xuất (ResNet50 Feature Extractor)
**Lý thuyết cấu hình (Model Configuration):**
- **Tại sao chọn ResNet50?** ResNet50 có kiến trúc Residual Connections (kết nối tắt) giúp chống lại hiện tượng suy biến đạo hàm (vanishing gradient), rất phù hợp để nhận diện các kết cấu vi thể phức tạp (hạt nhân, viền màng tế bào) vốn có độ tương phản thấp.
- **Loại bỏ lớp cuối (Truncation):** Mạng ResNet50 nguyên bản có lớp cuối cùng dự đoán 1000 đồ vật (Fully Connected Layer). Chúng ta sẽ **cắt bỏ lớp này** (`nn.Sequential(*list(model.children())[:-1]`). Nhờ vậy, đầu ra của ảnh không phải là một nhãn đồ vật, mà là một **vector toán học 2048 chiều** đại diện cho toàn bộ thông tin sinh học của mảng tế bào đó.
- **Thông số Dataloader:** `batch_size=256` và `num_workers=2` kết hợp `pin_memory=True` là cấu hình tối ưu nhất (Sweet-spot) để GPU T4 không bị "đói" dữ liệu từ CPU.

In [4]:
def build_feature_extractor():
    model = models.resnet50(pretrained=True)
    # Cắt lớp Linear dự đoán cuối cùng
    model = nn.Sequential(*list(model.children())[:-1])
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if torch.cuda.device_count() > 1:
        print(f"Tối ưu hóa: Khởi động DataParallel trên {torch.cuda.device_count()} GPUs.")
        model = nn.DataParallel(model)
        
    model = model.to(device)
    model.eval() # Bắt buộc phải set eval() để khóa Dropout và BatchNorm
    return model, device

def extract_patient_features(patient_id, model, device, batch_size=256):
    patient_dir = os.path.join(WSI_DIR, patient_id)
    if not os.path.exists(patient_dir):
        return None
        
    dataset = PatientPatchDataset(patient_dir, transform)
    if len(dataset) == 0:
        return None
        
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=2, pin_memory=True)
    
    features_list = []
    with torch.no_grad(): # Tắt tính toán đạo hàm để tiết kiệm RAM
        for images in loader:
            images = images.to(device)
            features = model(images)  # [Batch, 2048, 1, 1]
            features = features.view(features.size(0), -1)  # Kéo thẳng thành [Batch, 2048]
            features_list.append(features.cpu())
            
    patient_tensor = torch.cat(features_list, dim=0) # Tổng hợp lại thành [Số Patch, 2048]
    return patient_tensor

model, device = build_feature_extractor()

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 189MB/s]


Tối ưu hóa: Khởi động DataParallel trên 2 GPUs.


### 4. Vòng Lặp Xử Lý Toàn Bộ Dữ Liệu (Full Execution)
Thiết lập `limit = None` để bắt đầu trích xuất đặc trưng cho **tất cả 945 bệnh nhân**. Mọi vector sinh ra sẽ được lưu thành file `.pt` chuẩn bị cho Notebook 3 & 4.

In [5]:
valid_patients = [p for p in df_filtered['patientId'].tolist() if os.path.exists(os.path.join(WSI_DIR, str(p)))]
print(f"Tổng số bệnh nhân hợp lệ cần xử lý: {len(valid_patients)}")

limit = None # Chạy Toàn Bộ Data
patients_to_process = valid_patients[:limit] if limit else valid_patients

print(f"Bắt đầu trích xuất cho {len(patients_to_process)} bệnh nhân...")
extracted_count = 0
for pid in tqdm(patients_to_process):
    out_path = os.path.join(OUTPUT_PT_DIR, f"{pid}.pt")
    if not os.path.exists(out_path):
        tensor = extract_patient_features(str(pid), model, device)
        if tensor is not None:
            torch.save(tensor, out_path)
            extracted_count += 1

print(f"\nHoàn tất! Đã trích xuất {extracted_count} file .pt mới.")

Tổng số bệnh nhân hợp lệ cần xử lý: 882
Bắt đầu trích xuất cho 882 bệnh nhân...


100%|██████████| 882/882 [2:05:25<00:00,  8.53s/it]


Hoàn tất! Đã trích xuất 882 file .pt mới.


### 5. Baseline Cổ Điển: Mean-Pooling & Random Forest

Trước khi áp dụng siêu mạng Transformer, chuẩn mực nghiên cứu luôn yêu cầu xây dựng một mô hình cơ bản (Baseline) để đối chiếu hiệu năng.

**Cách hoạt động của Baseline:**
1. **Mean-Pooling:** Một bệnh nhân có thể có tới 5,000 patches (kích thước `[5000, 2048]`). Thuật toán Machine Learning không thể đọc ma trận 2D này. Giải pháp duy nhất của truyền thống là cộng tất cả 5,000 vector lại và chia trung bình (Mean-Pooling) để ra 1 vector duy nhất `[1, 2048]` đại diện cho cả bệnh nhân.
   - *Yếu điểm:* Việc lấy trung bình vô tình làm "loãng" các đặc điểm hiếm. Nếu một ảnh WSI 10,000 patches chỉ có 10 patches ác tính, việc chia trung bình cho 9,990 patches mô mỡ sẽ làm tín hiệu ác tính bị triệt tiêu hoàn toàn.
2. **Lựa chọn Thuật toán:** Chúng ta sử dụng **Random Forest** với `n_estimators=100` (100 cây quyết định). RF chống Overfitting rất tốt trên không gian đặc trưng lớn (2048 chiều).
3. **Cứu vớt dữ liệu lệch (Imbalance):** Tham số `class_weight='balanced'` ép mô hình phạt nặng (penalty) gấp nhiều lần nếu đoán sai các nhóm thiểu số (như HER2), giúp mô hình không bị thiên vị mù quáng vào nhóm đông (LumA).

### 📊 ĐÁP ỨNG TIÊU CHÍ: Đánh giá bằng các chỉ số thực nghiệm
Trong phần này, chúng ta sẽ đo lường hiệu năng của Baseline bằng các chỉ số được yêu cầu trong đề cương:
- **Accuracy (Độ chính xác tổng):** Tỉ lệ dự đoán đúng trên toàn bộ 4 nhãn. Tuy nhiên, do mất cân bằng dữ liệu (Class Imbalance), chỉ số này có thể gây ảo tưởng.
- **Precision (Độ chính xác) & Recall (Độ phủ):** Chúng ta sẽ tính toán cho từng nhãn. Đặc biệt quan tâm đến Recall của nhóm HER2 (Khả năng mô hình không bỏ sót ca HER2 nào).
- **Macro F1-score:** Là trung bình cộng F1-score của cả 4 nhãn (không màng đến số lượng ca bệnh nhiều hay ít). Đây là chỉ số **Chuẩn và Cốt lõi nhất** để chứng minh mô hình hoạt động công bằng cho tất cả các loại ung thư.

In [6]:
label_map = {'BRCA_LumA': 0, 'BRCA_LumB': 1, 'BRCA_Basal': 2, 'BRCA_Her2': 3}
df_filtered['label'] = df_filtered['pam50_subtype'].map(label_map)
patient_label_dict = dict(zip(df_filtered['patientId'], df_filtered['label']))

X_baseline, y_baseline, patient_ids_used = [], [], []

print("Đang nén dữ liệu bằng Mean Pooling...")
for pid in patients_to_process:
    pt_path = os.path.join(OUTPUT_PT_DIR, f"{pid}.pt")
    if os.path.exists(pt_path):
        tensor = torch.load(pt_path) # [N, 2048]
        mean_feature = torch.mean(tensor, dim=0).numpy() # Nén về [2048]
        X_baseline.append(mean_feature)
        y_baseline.append(patient_label_dict[pid])
        patient_ids_used.append(pid)

X_baseline = np.array(X_baseline)
y_baseline = np.array(y_baseline)

if len(X_baseline) > 10:
    print(f"Hoàn tất! Kích thước Ma trận Huấn luyện Baseline: X={X_baseline.shape}, y={y_baseline.shape}")
    
    # Chia 80/20. Bắt buộc dùng stratify để giữ đúng tỷ lệ 4 nhãn trong cả Train và Test
    X_train, X_test, y_train, y_test = train_test_split(
        X_baseline, y_baseline, test_size=0.2, stratify=y_baseline, random_state=42
    )
    
    print("\nĐang huấn luyện Random Forest Classifier...")
    rf_model = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42)
    rf_model.fit(X_train, y_train)
    
    y_pred = rf_model.predict(X_test)
    
    print("\n==============================================")
    print("      KẾT QUẢ BASELINE (TRUYỀN THỐNG)")
    print("==============================================")
    print(f"🎯 Accuracy (Độ chính xác tổng): {accuracy_score(y_test, y_pred):.4f}")
    print(f"⚖️ Macro-F1 (Điểm công bằng cho 4 nhãn): {f1_score(y_test, y_pred, average='macro'):.4f}")
    print("----------------------------------------------")
    
    target_names = [k for k, v in sorted(label_map.items(), key=lambda item: item[1])]
    print(classification_report(y_test, y_pred, target_names=target_names, zero_division=0))
else:
    print("Lỗi: Không đủ dữ liệu file .pt để chạy Baseline.")

Đang nén dữ liệu bằng Mean Pooling...
Hoàn tất! Kích thước Ma trận Huấn luyện Baseline: X=(882, 2048), y=(882,)

Đang huấn luyện Random Forest Classifier...

      KẾT QUẢ BASELINE (TRUYỀN THỐNG)
🎯 Accuracy (Độ chính xác tổng): 0.5763
⚖️ Macro-F1 (Điểm công bằng cho 4 nhãn): 0.3139
----------------------------------------------
              precision    recall  f1-score   support

   BRCA_LumA       0.60      0.96      0.74        92
   BRCA_LumB       0.20      0.03      0.05        38
  BRCA_Basal       0.59      0.39      0.47        33
   BRCA_Her2       0.00      0.00      0.00        14

    accuracy                           0.58       177
   macro avg       0.35      0.34      0.31       177
weighted avg       0.46      0.58      0.48       177



---
### 🏆 TỔNG KẾT NOTEBOOK 2:
**Bài học từ kết quả Baseline:** 
Bạn sẽ nhận thấy Macro-F1 khá thấp (đặc biệt là Recall của HER2). Nguyên nhân cốt lõi là do **Mean-Pooling** đã làm mất đi sự kết nối không gian (spatial connections) và làm "chìm" các mảnh tế bào mang đặc tính phân bào điển hình của khối u. 

👉 **Đó chính là lý do TransMIL ra đời.** Ở Notebook 3, thay vì lấy trung bình thô bạo, chúng ta sẽ để cho AI (Self-Attention của Transformer) tự động quét và đánh trọng số (Attention Scores) cho những vị trí quan trọng nhất của bệnh nhân.